
# CLD data dictionary

checking data dictionary, and if the variable names match the vensim models

last update: 06/19/2024


In [250]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# READ I/O

In [248]:
from spellchecker import SpellChecker
from pathlib import Path

import numpy as np
import os 
import pickle
import pandas as pd
import sys
import networkx as nx
import matplotlib.pyplot as plt
import gravis as gv
import json

In [249]:

# Get the current working directory
cwd = Path.cwd()

# Get the path to the parent directory
parent_dir = cwd.parent

# Add the parent directory to the Python path
sys.path.append(str(parent_dir))

# Now you can import the data_io module from src
import src.data_io as data_io
import src.cld_nx as cld_nx
import src.parse_cld as parse_cld

from src.vensim_parser import CLDParser

# Read Data

In [247]:
print(os.getcwd())


/Users/gl665/PycharmProjects/Phrame_MentalMaps/notebook


## read raw data

In [256]:
# SYNCH 
dataset_folder_path = 'results/SYNCH/JSON_file/RC/clean'

# PHRAME
# dataset_folder_path = 'results/PHRAME/JSON_file/RC'

In [252]:
# Go one level up from the current directory
cwd = os.getcwd()

parent_dir = Path(cwd).parent
print(parent_dir)

my_data= {}
folder_path = str(parent_dir / dataset_folder_path)


cld_data = data_io.read_json(folder_path, my_data)


/Users/gl665/PycharmProjects/Phrame_MentalMaps
load: DSS full clean_edited.json
load: Passages Clean_edited.json
load: Wavi combined clean_edited.json
load: full circle clean_edited.json


# Read Data Definition Dictionary

In [251]:
input_file = 'data/SYNCH/RC/excel_sheets/Rapid City SYNCH Models data dictionary.xlsx'

# Data Dictionary 
# Go one level up from the current directory
cwd = os.getcwd()


file_path = str(parent_dir / input_file)

# Read all sheets into a dictionary of DataFrames
excel_data = pd.read_excel(file_path, sheet_name=None)

# Print the keys (sheet names) of the dictionary
print("Sheet names:", list(excel_data.keys()))


# print(f"\nData from '{sheet_name}' sheet:\n{df.head()}")

Sheet names: ['full', 'Passages', 'WAVI (combined)', 'Full Circle (patient combined)', 'DSS (combined)', 'justice- drugs', 'Justice - IPV', 'full circle (provider)', 'E model 5 7 2020', 'FM 5 122020', 'nat model', 'j model 7 13 20', 'c model 7 6 20', 'reentry', 'RC Diversion', 'RC Clean', 'RC Legal', 'RC Full Circle', 'description of sessions', 'Sheet1']


In [278]:
#TODO: change this based on the cld I am checking 

my_cld_sheet_name = "Full Circle (patient combined)"
my_json_file_name = "full circle clean_edited.json"


In [279]:
# Access a specific sheet
df = excel_data[my_cld_sheet_name]

print(f"\nData from '{my_cld_sheet_name}':")
df.head()


Data from 'Full Circle (patient combined)':


,variable,changed variable name,definition,notes,final_name
0,Abiility to fufill DSS requests,ability to fulfill child welfare system requir...,having the adequate internal and external capa...,NaN,ability to fulfill child welfare system requir...
1,Accountability,NaN,Being held responsible for one's actions by ot...,NaN,accountability
2,ACES,adverse childhood experiences,Adverse childhood experiences - can include ch...,NaN,adverse childhood experiences
3,Alcohol use,alcohol misuse,using alcohol - in reference to misuse,similar loop btwn alcohol and negative associa...,alcohol misuse
4,Attitudes towards DSS,personal positive attitude towards the child w...,ones' personal attitudes (here as positive) to...,NaN,personal positive attitude towards the child w...


# matching the excel sheet with the vensim file 

In [280]:
names_from_excel = df["changed variable name"].fillna(df["variable"]).tolist()

df['final_name'] = df["changed variable name"].fillna(df["variable"]).str.lower()


print("Numbers of variables in the excel file: ", len(names_from_excel))
# print(names_from_excel)

Numbers of variables in the excel file:  77


In [281]:
df.head(4)

,variable,changed variable name,definition,notes,final_name
0,Abiility to fufill DSS requests,ability to fulfill child welfare system requir...,having the adequate internal and external capa...,NaN,ability to fulfill child welfare system requir...
1,Accountability,NaN,Being held responsible for one's actions by ot...,NaN,accountability
2,ACES,adverse childhood experiences,Adverse childhood experiences - can include ch...,NaN,adverse childhood experiences
3,Alcohol use,alcohol misuse,using alcohol - in reference to misuse,similar loop btwn alcohol and negative associa...,alcohol misuse


In [282]:
# dict_cld = {name_file: nxgraph}

dict_cld = {}

for k, v in my_data.items():
    cld_list, G_cld = cld_nx.get_cld_graph(my_data, k)
    # cld_name = dict_file_rename.get(k, "")
    dict_cld[k] = G_cld

for k, v in dict_cld.items(): 
    print(k, ":", v)

DSS full clean_edited.json : DiGraph with 85 nodes and 181 edges
Passages Clean_edited.json : DiGraph with 18 nodes and 61 edges
Wavi combined clean_edited.json : DiGraph with 35 nodes and 78 edges
full circle clean_edited.json : DiGraph with 77 nodes and 182 edges


In [283]:
cld_var_names  = list(dict_cld[my_json_file_name].nodes())
# cld_var_names

# check if the data dictionary and the CLD data has the exact same name

In [284]:
# check how much variables overlapped between the two list of all variable names 

name_list = df['final_name'].tolist()
overlap = [var for var in cld_var_names if var.lower() in name_list]
print("Overlapping variables between all the variables names used in the RC models and the handcraft model: ", len(overlap))


for i, name in enumerate(overlap, start=1):
    print(f"{i}. {name}")



Overlapping variables between all the variables names used in the RC models and the handcraft model:  77
1. intimate partner violence
2. police racial bias and trauma sensitivity skills
3. partner dependence
4. fear of violent partner
5. relationship functioning
6. healthy support system
7. alcohol misuse
8. drug misuse
9. childhood home functioning
10. mental health
11. coping skills
12. accountability
13. generational trauma
14. experiencing indigenous culture
15. indigenous cultural identity
16. empowerment groups that are focused on indigenous culture
17. healthy parenting practices
18. using violence in self-defense from violent partner
19. fear of getting arrested
20. being drunk during intimate partner violence event
21. fear of children being taken by the child welfare system
22. adverse childhood experiences
23. children learning parenting skills
24. children living in home
25. healthy relationship between children and parents
26. drug use during intimate partner violence even

In [285]:

# check not overlapped variables in cld not in excel sheet
not_overlap = [var for var in cld_var_names if var.lower() not in df['final_name'].tolist()]


for count, i in enumerate(not_overlap):
    print(count, i)

In [286]:
# check not overlapped variables in excel sheet but not in cld 
not_overlap = [var for var in df['final_name'] if var.lower() not in cld_var_names]


for count, i in enumerate(not_overlap):
    print(count, i)

# Create Label data

In [287]:
# get dictionary for df['final_name'] and df['definition']
dict_name_def = dict(zip(df['definition'],df['final_name']))

folder_path = str(parent_dir / 'results/SYNCH/JSON_file/RC/results sheets/data_dictionary/'/my_json_file_name )

with open(folder_path, 'w') as f:
    json.dump(dict_name_def, f)


In [268]:
# dict_name_def = dict(zip(df['final_name'], df['Definition']))
# file_name = 'data_dictionary_passages.json'

# folder_path = str(parent_dir / 'results/SYNCH/JSON_file/RC/results sheets/data_dictionary/'/file_name )

# with open(folder_path, 'w') as f:
#     json.dump(dict_name_def, f)